# Source text

![image_1778508668540.png](./image_1778508668540.png "image_1778508668540.png")

# Install libraries

In [0]:
# Reading PDF
%pip install pypdf

# Chunking
%pip install langchain-text-splitters

# Agent 
%pip install -U -qqqq databricks-openai databricks-agents

# Embeddings
%pip install sentence-transformers

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Restart the runtime
dbutils.library.restartPython()

# PDF

In [0]:

pdf_path = "/Volumes/agent/default/rag_volume/rag_file.pdf"

from pypdf import PdfReader

reader = PdfReader(pdf_path)
text = "\n".join(page.extract_text() or "" for page in reader.pages)

print(text[:500])


The Czech Republic, also called Czechia, is a landlocked country in Central Europe. It borders Austria, 
Germany, Poland, and Slovakia. The country has a mostly hilly landscape and a temperate climate. 
The capital and largest city is Prague, followed by other important cities such as Brno, Ostrava, Plzeň, 
and Liberec. 
The Czech lands have a long history. The Duchy of Bohemia was established in the late 9th century 
and later became a kingdom. In the 16th century, the region came under the rul


# Chunking

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=150,
)

chunks = splitter.split_text(text)

print(len(chunks))
print(chunks[0][:500])


/local_disk0/.ephemeral_nfs/envs/pythonEnv-6ff014fc-dfa9-4108-aee9-0f9ddb313ab7/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


5
The Czech Republic, also called Czechia, is a landlocked country in Central Europe. It borders Austria, 
Germany, Poland, and Slovakia. The country has a mostly hilly landscape and a temperate climate. 
The capital and largest city is Prague, followed by other important cities such as Brno, Ostrava, Plzeň, 
and Liberec.


# Embeddings

### Create Embeddings

In [0]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # open-source, zdarma
embeddings = model.encode(chunks, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

### Save embeddings into the table

In [0]:
rows = [
    (str(i), chunks[i], embeddings[i].tolist())
    for i in range(len(chunks))
]

df = spark.createDataFrame(
    rows,
    ["chunk_id", "chunk_text", "embedding"]
)

df.write.mode("overwrite").saveAsTable("rag_emb")

### Reading embeddings from Delta into the RAM (Pandas) -> faster

In [0]:
df = spark.table("rag_emb")
pdf_df = df.toPandas()

display(pdf_df)


chunk_id chunk_text embedding 0 The Czech Republic, also called Czechia, is a landlocked country in Central Europe. It borders Austria, 
Germany, Poland, and Slovakia. The country has a mostly hilly landscape and a temperate climate. 
The capital and largest city is Prague, followed by other important cities such as Brno, Ostrava, Plzeň, 
and Liberec. List(0.07651429623365402, 0.045982230454683304, -0.0619969516992569, 0.03137335926294327, 0.05949870124459267, -0.07999139279127121, -0.015027214772999287, 0.0013256222009658813, -0.036943014711141586, -0.04372244328260422, 0.0022664638236165047, -0.062111273407936096, -0.06775975227355957, -0.04141491651535034, -0.07343398034572601, -0.009160937741398811, -0.016521183773875237, 0.03893263265490532, -0.03729018196463585, 0.04385622590780258, -0.01237823162227869, -0.06498904526233673, 0.07450252771377563, 0.005359783302992582, -0.008828078396618366, -0.01130120363086462, 0.04045823961496353, 0.06553111970424652, 0.026520052924752235, -0.006887299474328756, 0.03572704270482063, 0.05956416204571724, -0.03314173221588135, 0.020923513919115067, 0.004740833304822445, -0.014216212555766106, -0.111335389316082, -0.040676575154066086, 0.042746979743242264, 0.029745593667030334, 0.04461399465799332, -0.036325205117464066, -0.008849025703966618, 0.06764201074838638, 0.024761928245425224, 0.06528148800134659, -0.012152031064033508, 0.03554212301969528, -0.030356954783201218, -0.0063156126998364925, 0.06821136176586151, -0.06314513087272644, -0.013474724255502224, 0.07786565274000168, -0.10612626373767853, 0.032141104340553284, -0.06016245484352112, 0.07088596373796463, 0.010098341852426529, 0.01014812383800745, 0.028682641685009003, -1.9695547234732658E-4, -0.029013026505708694, 0.020997928455471992, 0.008832254447042942, -0.0034886635839939117, -0.02548135258257389, 0.045770373195409775, -0.05670955777168274, -0.05254427716135979, 0.027747584506869316, -0.06170622631907463, 0.012529305182397366, 0.04414384439587593, -0.008689340204000473, -0.08309905976057053, -0.07854384928941727, 0.04451693966984749, -7.604018901474774E-4, 0.021574784070253372, 0.04052512347698212, 0.02441052347421646, -0.08208689838647842, 8.322073263116181E-4, -0.023626910522580147, 0.030940797179937363, -0.02870943769812584, 0.012168827466666698, 0.038308285176754, -0.018944790586829185, 0.012418581172823906, 0.06533816456794739, 0.06305604428052902, 0.01559763215482235, -0.02694813348352909, 0.047695741057395935, 0.10475771874189377, 0.03494357690215111, 0.01887969858944416, 0.03956184163689613, 0.05606028065085411, 0.026621786877512932, 0.09220507740974426, 0.01266466174274683, -0.02436687983572483, 0.01939903199672699, 0.057995714247226715, -0.0033290470018982887, -0.06792499125003815, -0.006755897309631109, -0.06629471480846405, 0.00762950861826539, -0.04985358566045761, -0.12090934067964554, 0.043201617896556854, -0.044880155473947525, 0.11218024045228958, -0.048732876777648926, 0.03791036456823349, -0.005528726615011692, -0.031229961663484573, -0.06068511679768562, -0.007013019174337387, 0.001486181397922337, 0.12218951433897018, 0.05522202327847481, -0.011564880609512329, -2.775094022780802E-33, -0.012231696397066116, -0.04812118038535118, -0.015568225644528866, 0.011078345589339733, -0.10070645809173584, 0.061416998505592346, 0.02696121856570244, 0.005530787166208029, -0.07192894071340561, 0.025501061230897903, -0.07775600999593735, -0.09270156174898148, -0.019971149042248726, -0.035852160304784775, 0.06434379518032074, 0.09650114178657532, 0.03156280145049095, -0.0034557650797069073, -0.07161129266023636, 0.10818153619766235, 0.0156776811927557, 0.030515309423208237, 0.042756907641887665, 0.006323646754026413, 0.002303800778463483, -0.028046440333127975, 0.022052600979804993, -0.03581772372126579, -0.010389472357928753, -0.03674021735787392, -0.00316570233553648, 0.06582068651914597, -0.04978315904736519, -0.03730165958404541, 0.03611933812499046, 0.017495347186923027, -0.052110955119132996, 0.043439902365207

In [0]:
import numpy as np

doc_embeddings = np.array(pdf_df["embedding"].tolist())
chunks = pdf_df["chunk_text"].tolist()

# LLM

In [0]:
import mlflow
from databricks.sdk import WorkspaceClient

mlflow.openai.autolog()

openai_client = WorkspaceClient().serving_endpoints.get_open_ai_client()

In [0]:
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"

### Prompt function

In [0]:
import numpy as np

def cosine_similarity(a, b):   
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [0]:
from sentence_transformers import SentenceTransformer

def rag_answer(query: str, top_k: int = 3) -> str:
    # 1) embedding dotazu
    model = SentenceTransformer("all-MiniLM-L6-v2")
    query_embedding = model.encode(query)

    # 2) similarity search nad uloženými embeddingy
    scores = np.array([
        cosine_similarity(query_embedding, emb)
        for emb in doc_embeddings
    ])

    top_indices = scores.argsort()[-top_k:][::-1]
    context_chunks = [chunks[i] for i in top_indices]
    context = "\n\n".join(context_chunks)

    # 3) RAG prompt pro LLM
    messages = [
        {
            "role": "system",
            "content": (
                "Answer ONLY using the provided context. "
                "If the answer is not in the context, say 'I don't know'."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion:\n{query}",
        },
    ]

    # 4) volání LLM
    response = openai_client.chat.completions.create(
        model=LLM_ENDPOINT_NAME,
        messages=messages,
    )

    return response.choices[0].message.content

### Question - test (correct)

In [0]:
rag_answer("When did the Nazis occupied Czechoslovakia?")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

'In 1938, Nazi Germany occupied Czechoslovakia.'

Trace(trace_id=tr-75e6f54474f43192fc9156653054c8d8)

In [0]:
rag_answer("When did Czechoslovakia split?")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

'Czechoslovakia split into two independent countries in 1993.'

Trace(trace_id=tr-55b3d8a1413d1ce1b24e833b834d078d)

### Question outside the context of the document

In [0]:
rag_answer("How much milk do I need for a cake?")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

"I don't know."

Trace(trace_id=tr-ef65da4d0e7b849bee70715d47142925)